### ⚠️ Patched for Local Execution
This notebook was originally designed for Google Colab. It has been automatically patched:
- Google Colab-specific imports and `drive.mount()` calls have been commented out
- Colab file paths (`/content/drive/...`) have been replaced with relative paths (`./`)
- `!pip install` commands have been commented out (install packages in your venv instead)

**To run locally:** activate your Python virtual environment first, then run this notebook in VS Code or Jupyter.

In [ ]:
# [PATCHED] from google.colab import drive
# [PATCHED] drive.mount('./')

In [ ]:
working_folder='./'

photos_folder=working_folder + 'photos/'

In [ ]:
import pandas as pd

csv_file_path = photos_folder + 'fer2013.csv'
fer_df = pd.read_csv(csv_file_path)

fer_df.head()

In [ ]:
print(fer_df.shape)

In [ ]:
import numpy as np

def process_pixels(pixel_string):

    lst=list(map(int, pixel_string.split()))

    pixels = np.array(lst, dtype=np.uint8)
    return pixels

In [ ]:
fer_df['pixels_array'] = fer_df['pixels'].apply(process_pixels)

In [ ]:
unique_emotions = fer_df['emotion'].unique()

unique_emotions

In [ ]:
fer_df_training=fer_df[fer_df['Usage']=='Training']

emotion_counts = fer_df_training['emotion'].value_counts()
emotion_counts

In [ ]:
min_emotion_training_count = emotion_counts.min()

min_emotion_training_count

In [ ]:
fer_df_val=fer_df[fer_df['Usage']=='PublicTest']
emotion_counts = fer_df_val['emotion'].value_counts()
min_emotion_val_count = emotion_counts.min()

min_emotion_val_count

In [ ]:
fer_df_testing=fer_df[fer_df['Usage']=='PrivateTest']
emotion_counts = fer_df_testing['emotion'].value_counts()
min_emotion_testing_count = emotion_counts.min()

min_emotion_testing_count

In [ ]:
selected_rows = []

for emotion in unique_emotions:
    emotion_rows = fer_df[(fer_df['emotion'] == emotion) & (fer_df['Usage'] == 'Training')]

    random_indices = np.random.choice(emotion_rows.index, size=2*min_emotion_training_count, replace=True)

    selected_rows.extend(random_indices)

fer_train_df = fer_df.loc[selected_rows]


selected_rows = []

for emotion in unique_emotions:
    emotion_rows = fer_df[(fer_df['emotion'] == emotion) & (fer_df['Usage'] == 'PublicTest')]
    random_indices = np.random.choice(emotion_rows.index, size=2*min_emotion_val_count, replace=True)
    selected_rows.extend(random_indices)

fer_val_df = fer_df.loc[selected_rows]

selected_rows = []
for emotion in unique_emotions:
    emotion_rows = fer_df[(fer_df['emotion'] == emotion) & (fer_df['Usage'] == 'PrivateTest')]
    random_indices = np.random.choice(emotion_rows.index, size=2*min_emotion_testing_count, replace=True)
    selected_rows.extend(random_indices)

fer_test_df = fer_df.loc[selected_rows]

In [ ]:
fer_train_df.reset_index(drop=True, inplace=True)
fer_test_df.reset_index(drop=True, inplace=True)
fer_val_df.reset_index(drop=True, inplace=True)

In [ ]:
print(len(fer_train_df))
print(len(fer_val_df))
print(len(fer_test_df))

In [ ]:
pixels_array0=fer_train_df.loc[0,'pixels_array']

pixels_array0

In [ ]:
image0 = np.reshape(pixels_array0, (48, 48))

image0

In [ ]:
image0_rgb = image0[..., np.newaxis]

image0_rgb

In [ ]:
image0_rgb.shape

In [ ]:
image0_rgb = np.repeat(image0_rgb, 3, axis=2)

image0_rgb

In [ ]:
def prepare_fer_data(df):

    image_list = []

    image_labels = list(map(int, df['emotion']))

    for index,row in df.iterrows():

        pixels_array=row['pixels_array']

        image = np.reshape(pixels_array, (48, 48))

        image = image[..., np.newaxis]

        image = np.repeat(image, 3, axis=2)

        image = image.astype(int).tolist()

        image_list.append(image)

    output_df = pd.DataFrame(list(zip(image_list, image_labels)),
               columns =['img', 'label'])

    return output_df

In [ ]:
fer_train_df = prepare_fer_data(fer_train_df)
fer_val_df = prepare_fer_data(fer_val_df)
fer_test_df = prepare_fer_data(fer_test_df)

In [ ]:
fer_train_df.head()

In [ ]:
# [PATCHED] !pip install datasets

In [ ]:
from datasets import *

In [ ]:
train_ds = Dataset.from_pandas(fer_train_df)

train_ds

In [ ]:
val_ds = Dataset.from_pandas(fer_val_df)

val_ds

In [ ]:
test_ds = Dataset.from_pandas(fer_test_df)
test_ds

In [ ]:
train_ds.save_to_disk(working_folder  + 'train_dataset')
val_ds.save_to_disk(working_folder + 'val_dataset')
test_ds.save_to_disk(working_folder +'test_dataset')